In [ ]:
import fitz  # PyMuPDF
import re

def extract_and_clean_pdf(pdf_path, output_txt_path="the_lotr.txt", start_page=0, end_page=None):
    
    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    if end_page is None or end_page > total_pages:
        end_page = total_pages

    print(f"Loading PDF with {total_pages} pages. Extracting from page {start_page} to {end_page}...")

    extracted_pages = []
    for page_num in range(start_page, end_page):
        page = doc[page_num]
        text = page.get_text()
        extracted_pages.append(text)

    full_text = "\n".join(extracted_pages)

    # initial cleaning 
    full_text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', full_text)

    full_text = full_text.replace('\t', ' ')

    full_text = re.sub(r'[ ]+', ' ', full_text)

    full_text = re.sub(r'\n{3,}', '\n\n', full_text)

    # save in txt file 
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(full_text)

    print(f"Extraction finished! Saved to {output_txt_path}")
    print(f"Total characters: {len(full_text):,}")
    return full_text

In [ ]:
from importlib.metadata import version
import torch

print("torch version:", version("torch"))


from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference

torch version: 2.11.0+cu128


In [ ]:
def split_text_interleaved(text, num_blocks=10, train_ratio=0.9):
    
    block_size = len(text) // num_blocks
    train_chunks = []
    val_chunks = []

    for i in range(num_blocks):
        start = i * block_size
        end = (i + 1) * block_size if i < num_blocks - 1 else len(text)
        block = text[start:end]

        # split point based on ratio
        split_point = int(len(block) * train_ratio)

        # find nearest spase for preventing from word cutting in the border of split 
        split_point = block.rfind(" ", 0, split_point)
        if split_point == -1:
            split_point = int(len(block) * train_ratio)

        train_chunks.append(block[:split_point].strip())
        val_chunks.append(block[split_point:].strip())

    train_text = "\n\n".join(train_chunks)
    val_text = "\n\n".join(val_chunks)

    return train_text, val_text

In [ ]:
import os
import torch
import tiktoken
from utils import GPTDatasetV1, create_dataloader_v1  


txt_file = "lotr-full.txt"
output_txt_path = "lotr-full.txt"
txt_file1 = "lotr1.txt"
txt_file2 = "lotr2.txt"
txt_file3 = "lotr3.txt"
pdf_file1 = "1.pdf"
pdf_file2 = "2.pdf"
pdf_file3 = "3.pdf"
if not os.path.exists(txt_file):
    
    text_data = extract_and_clean_pdf(pdf_file1, output_txt_path=txt_file1, start_page=22, end_page=None)
    text_data += extract_and_clean_pdf(pdf_file2, output_txt_path=txt_file2, start_page=6, end_page=245)
    text_data += extract_and_clean_pdf(pdf_file3, output_txt_path=txt_file3, start_page=11, end_page=302)

    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(text_data)
else:
    with open(txt_file, "r", encoding="utf-8") as file:
        text_data = file.read()

# split train and validation 
train_ratio = 0.8

# intelligence spliting books into 10 blocks
train_text, val_text = split_text_interleaved(text_data, num_blocks=10, train_ratio=0.9)

print(f"Train characters: {len(train_text):,} (~{len(train_text)//4:,} tokens)")
print(f"Val characters:   {len(val_text):,} (~{len(val_text)//4:,} tokens)")

torch.manual_seed(123)

# create dataloaders
train_loader = create_dataloader_v1(
    train_text,
    batch_size=16,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=int(GPT_CONFIG_124M["context_length"]/4),
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_text,
    batch_size=16,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=int(GPT_CONFIG_124M["context_length"]/4),
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train characters: 2,232,063 (~558,015 tokens)
Val characters:   248,059 (~62,014 tokens)
Train batches: 595
Val batches: 66


In [ ]:
import math
import torch
from utils import evaluate_model, generate_and_print_sample, calc_loss_batch

def train_model(
    model, train_loader, val_loader, optimizer, device,
    n_epochs, eval_freq, eval_iter, start_context, tokenizer,
    warmup_steps, initial_lr=3e-05, min_lr=1e-6,
    peak_lr=None, accumulation_steps=4, save_path="best_model.pt", resume_path=None
):
    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen = 0
    global_step = -1
    start_epoch = 0
    best_val_loss = float("inf")
    # loading from the checkpoint based on the user request
    if resume_path and os.path.exists(resume_path):
        print(f"Loading checkpoint from: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        global_step = checkpoint["global_step"]
        best_val_loss = checkpoint.get("best_val_loss", float("inf"))
        tokens_seen = checkpoint.get("tokens_seen", 0)
        print(f"Resumed from Epoch {start_epoch}, Step {global_step}, Best Val Loss: {best_val_loss:.3f}")


    if peak_lr is None:
        peak_lr = optimizer.param_groups[0]["lr"]
    else:
        for param_group in optimizer.param_groups:
            param_group["lr"] = initial_lr

    # calculate optimizing steps
    steps_per_epoch = math.ceil(len(train_loader) / accumulation_steps)
    total_training_steps = steps_per_epoch * n_epochs

    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()  

        for batch_idx, (input_batch, target_batch) in enumerate(train_loader):
            # calculate loss and divide it on accumulation steps
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss = loss / accumulation_steps
            loss.backward()

            tokens_seen += input_batch.numel()

            
            is_accumulation_done = ((batch_idx + 1) % accumulation_steps == 0)
            is_last_batch = (batch_idx + 1 == len(train_loader))

            if is_accumulation_done or is_last_batch:
                global_step += 1

                # apply warmup + Cosine Decay
                if global_step < warmup_steps:
                    lr = initial_lr + global_step * lr_increment
                else:
                    progress = (global_step - warmup_steps) / max(1, (total_training_steps - warmup_steps))
                    lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

                for param_group in optimizer.param_groups:
                    param_group["lr"] = lr
                track_lrs.append(lr)

                # gradinent clipping after warmpup and before update
                if global_step > warmup_steps:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                # weight updating and gradient cleaning for the next cycle
                optimizer.step()
                optimizer.zero_grad()

                # periodical evaluation
                if global_step % eval_freq == 0:
                    train_loss, val_loss = evaluate_model(
                        model, train_loader, val_loader,
                        device, eval_iter
                    )
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    track_tokens_seen.append(tokens_seen)

                    # progress login and checkpoint saving
                    saved_indicator = ""
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        checkpoint_data = {
                            "epoch": epoch,
                            "global_step": global_step,
                            "model_state_dict": model.state_dict(),
                            "optimizer_state_dict": optimizer.state_dict(),
                            "best_val_loss": best_val_loss,
                            "tokens_seen": tokens_seen
                        }
                        torch.save(checkpoint_data, save_path)
                        saved_indicator = " -> [SAVED NEW BEST]"

                    print(
                        f"Ep {epoch+1} (Step {global_step:06d}): "
                        f"Train loss {train_loss:.3f}, "
                        f"Val loss {val_loss:.3f}, "
                        f"Difference : {(val_loss - train_loss):.3f},"
                        f"LR {lr:.2e}"
                    )

        latest_data = {
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "tokens_seen": tokens_seen
            }
        torch.save(latest_data, "last_checkpoint.pt")
        # generate sample text at the end of each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen, track_lrs

In [ ]:
import tiktoken


model = GPTModel(GPT_CONFIG_124M)
model.to(device)

peak_lr = 3e-4  # learning rate


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=peak_lr,
    weight_decay=0.05
)
tokenizer = tiktoken.get_encoding("gpt2")

train_losses, val_losses, tokens_seen, lrs = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    n_epochs=8,
    eval_freq=15,
    eval_iter=5,
    start_context="When Bilbo came to himself",
    tokenizer=tokenizer,
    warmup_steps=80,
    initial_lr=1e-5,
    min_lr=1e-5,
    peak_lr=peak_lr,
    accumulation_steps=4,
    save_path="best_model.pt",
    resume_path=None
)

In [ ]:
from utils import plot_losses
import matplotlib.pyplot as plt

n_epochs = 8
epochs_tensor = torch.linspace(1, n_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
plt.tight_layout()
plt.show()

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth")

model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.to(device)

start_context = "Gollum was tugging at Frodo’s cloak and hissing with fear and"
generate_and_print_sample(
            model, tokenizer, device, start_context
        )